## Step 0: 데이터 로드하기

- 미션: 제공된 sensor_anomaly.csv 파일을 로드하여 data라는 변수에 리스트의 리스트로 불러오세요.

- 배경
> "여러분은 공장 설비에서 1000초간 10개 센서로 측정한 데이터를 sensor_anomaly.csv 파일로 전달받았습니다. 이 데이터는 (1000행, 10열) 형태인 일종의 엑셀 파일입니다. 이 데이터에는 고장난 센서가 포함되어 있습니다. 수업 시간에 배운 파이썬 기본 문법을 이용해서 몇 번 센서가 고장난 것인지 파악하고 해당 센서의 데이터는 삭제하고자 합니다."

- 다음을 실행하여 데이터를 불러 옵니다.

In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=2, suppress=True) # 출력 옵션 설정
df = pd.read_csv("sensor_anomaly.csv")

print(f"\n원본 데이터 (처음 5행):")
display(df.head())
print(f"\n원본 데이터 (마지막 5행):")
display(df.tail())

data = df.to_numpy().tolist()


원본 데이터 (처음 5행):


,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10
0,80.496714,3.486174,120.841995,0.576151,9.882923,54.578553,2.515843,91.304639,1.753053,30.813840
1,79.536582,3.453427,120.314551,0.404336,9.137541,53.987882,1.997434,90.534220,1.709198,27.881544
2,81.465649,3.477422,120.087787,0.428763,9.727809,55.199661,1.969801,90.638687,1.739936,29.562459
3,79.398293,3.685228,119.982454,0.447114,10.411272,52.802481,2.241773,86.668561,1.667181,30.295292
4,80.738467,3.517137,119.849657,0.484945,9.260739,53.704280,2.107872,91.797108,1.834362,27.355440



원본 데이터 (마지막 5행):


,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10
995,80.867805,3.522741,118.843202,0.451961,10.127064,56.254692,2.278376,88.241183,1.865067,30.638866
996,78.929334,3.421568,120.895045,0.488275,10.794574,55.902033,2.102674,89.982650,1.806338,28.907415
997,79.087412,3.570139,121.098855,0.530189,10.757659,54.024808,2.534854,88.468434,1.698731,27.360062
998,79.554205,3.449628,120.683718,0.512195,9.403513,54.293093,2.125708,86.980830,1.701905,28.843780
999,81.433625,3.519145,120.860819,0.425068,10.597107,57.341984,1.800331,88.800962,1.849577,30.966583


## Step 1: 센서별 통계 계산 (Aggregation & axis)

먼저 각 센서(특성)가 어떤 분포를 갖는지 알아야 합니다. 1000개의 샘플 전체에 대해 각 센서별(열별) 평균과 표준편차를 계산합니다.

- 미션: data 배열을 이용해 10개 센서 각각의 평균(mean_per_sensor)과 표준편차(std_per_sensor)를 계산하세요.


In [ ]:
# 여기 코드를 작성하세요.
import math

# 1. 각 센서(열)의 평균 계산 (결과는 10개 평균)
mean_per_sensor = []

for j in range(len(data[0])):
    S = 0.0
    for i in range(len(data)):
        S = S + data[i][j]
    mean_per_sensor.append(S / len(data))

# 2. 각 센서(열)의 표준편차 계산 (결과는 10개 표준 편차)
std_per_sensor = []

for j in range(len(data[0])):
    S = 0.0
    for i in range(len(data)):
        S = S + (data[i][j] - mean_per_sensor[j])**2
    std_per_sensor.append( math.sqrt(S / len(data)) )


print(f"\n--- Step 1: 통계 계산 ---")
print(f"센서별 평균 : {mean_per_sensor}")
print(f"센서별 표준편차: {std_per_sensor}")


--- Step 1: 통계 계산 ---
센서별 평균 : [80.010787543426, 3.503129238631213, 120.35061815065627, 0.4978462329503824, 10.006810801213208, 54.94734653916979, 2.1994622268784583, 89.82197262769725, 1.7976896400366986, 30.057157105626032]
센서별 표준편차: [1.0049297548654312, 0.1015537695087695, 12.079343055653926, 0.04914572514389691, 0.5060693489629222, 1.865709525263132, 0.20614291772772825, 6.255408297331046, 0.09863528862220444, 1.417216715744614]



## Step 2: Z-Score 정규화 (Broadcasting)

센서마다 스케일이 다릅니다 (e.g., 온도는 80, 압력은 3.5). AI 모델은 스케일에 민감하므로, 모든 센서 데이터를 평균 0, 표준편차 1을 갖도록 표준화(Z-score 정규화)합니다.

- 수식: Z=(X − μ)/σ​ (X: 원본 데이터, μ: 평균, σ: 표준편차)

- 미션: (1000, 10) 형태의 data 배열과 (10,) 형태의 mean_per_sensor, std_per_sensor를 이용해 정규화된 data를 만드세요.


In [ ]:
# 여기 코드를 작성하세요.

# Z-Score 정규화
for j in range(len(data[0])):
    for i in range(len(data)):
        data[i][j] = (data[i][j] - mean_per_sensor[j]) / std_per_sensor[j]

# (검증) 정규화된 데이터의 평균은 0에, 표준편차는 1에 매우 가까워야 함
print(f"정규화 데이터의 평균 (0에 근사):\n",np.array(data).mean(axis=0))
print(f"정규화 데이터의 표준편차 (1에 근사):\n",np.array(data).std(axis=0))

정규화 데이터의 평균 (0에 근사):
 [ 0.  0. -0.  0. -0.  0. -0. -0. -0. -0.]
정규화 데이터의 표준편차 (1에 근사):
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Step 3: 이상치 탐지 및 개수 확인 (Boolean Indexing)

정규화된 데이터(Z-score)에서 값이 +6 또는 -6을 벗어나면, 이는 통계적으로 '6-sigma'를 벗어나는 극단적인 이상치로 간주할 수 있습니다.

- 미션: data에서 절댓값이 6을 초과하는(즉, 6보다 크거나 -6보다 작은) 이상치가 총 몇 개인지 찾아내세요.


- 고장으로 판단되는 센서는 몇번과 몇번입니까?
    


In [ ]:
# 여기 코드를 작성하세요.
num_abnormal = 0 # 이상치 개수를 기록
sensors = [] # 이상치를 포함하는 센서 번호

for j in range(len(data[0])):
    for i in range(len(data)):
        if data[i][j] > 6 or data[i][j] < -6:
            num_abnormal = num_abnormal + 1
            sensors.append(j)

print("이상치의 개수: ", num_abnormal)
print("고장난 센서 인덱수: ", sensors)


이상치의 개수:  2
고장난 센서 인덱수:  [2, 7]


## Step 4: 고장 센서 식별 및 데이터셋에서 제거

Step 3에서 찾은 "고장 센서" 정보를 바탕으로, 해당 센서의 열(column) 전체를 원본 data에서 제거하여 깨끗한 데이터셋 cleaned_data를 만드세요. cleaned_data는 1000행, 8열이 되어야 합니다.



In [ ]:
cleaned_data = []

for i in range(len(data)):
    row = []
    for j in range(len(data[0])):
        # 고장난 센서 인덱스인 2와 7을 제외하고,
        # 정규화된 데이터를 다시 원본 스케일로 복원하여 추가
        if j != 2 and j != 7:
            original_value = (data[i][j] * std_per_sensor[j]) + mean_per_sensor[j]
            row.append(original_value)
    cleaned_data.append(row)

# 결과 확인
print(f"cleaned_data 크기: {len(cleaned_data)}행, {len(cleaned_data[0])}열 (고장 센서 2개 제거 완료)")
print("복원된 상위 1행 데이터 예시:")
print(cleaned_data[0])

cleaned_data 크기: 1000행, 8열 (고장 센서 2개 제거 완료)
복원된 상위 1행 데이터 예시:
[80.49671415301123, 3.4861735698828817, 0.5761514928204012, 9.882923312638331, 54.57855347749148, 2.5158425631014785, 1.753052561406505, 30.813840065378947]
